# LungCT 4DCT pTVPrep Visual Audit

Canonical preprocessing audited here: pTVReg fixed/moving and crop-index convention, CT clipping/scaling, 1 mm target shape, trilinear image resampling, and paired lung masks transformed through the same corrected crop for QA.

In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from matplotlib.colors import ListedColormap


PROTOCOL_NAME = 'pTVReg_iso1mm'
RUN_PREPROCESS_IF_MISSING = False


def first_existing(*paths: str) -> Path:
    for value in paths:
        path = Path(value)
        if path.exists():
            return path
    return Path(paths[0])


REPO_ROOT = first_existing('/homebase/code_sync/inrorspline', '/home/hengjie/code_sync/inrorspline')
DATA_ROOT = first_existing('/database/datasets/lungCT/DIR-Lab', '/data/hengjie/datasets/lungCT/DIR-Lab')
MASK_ROOT = DATA_ROOT / 'processed' / 'lungmask_R231_native'
PTVPREP_ROOT = DATA_ROOT / 'processed' / PROTOCOL_NAME
SUMMARY_PATH = PTVPREP_ROOT / '4DCT' / 'summary.json'
FIGURE_DIR = PTVPREP_ROOT / '4DCT' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not SUMMARY_PATH.exists() and RUN_PREPROCESS_IF_MISSING:
    cmd = [
        'python',
        str(REPO_ROOT / 'scripts/data/preprocessing/lungct/generate_4dct_ptvprep.py'),
        '--data-root', str(DATA_ROOT),
        '--mask-root', str(MASK_ROOT),
        '--out-root', str(PTVPREP_ROOT),
        '--case-ids', '1-10',
        '--device', 'cpu',
    ]
    subprocess.run(cmd, check=True)

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f'Missing preprocessing summary: {SUMMARY_PATH}')

summary = json.loads(SUMMARY_PATH.read_text())
rows = summary['rows']
print('repo', REPO_ROOT)
print('data', DATA_ROOT)
print('ptvprep', PTVPREP_ROOT)
print('cases', len(rows), 'all_verification_pass', summary['all_verification_pass'])


In [ ]:
def resolve_saved_path(path_text: str) -> Path:
    path = Path(path_text)
    if path.exists():
        return path
    text = str(path)
    replacements = [
        ('/database/', '/data/hengjie/'),
        ('/homebase/', '/home/hengjie/'),
    ]
    for prefix, repl in replacements:
        if text.startswith(prefix):
            alt = Path(text.replace(prefix, repl, 1))
            if alt.exists():
                return alt
    return path


def load_case_arrays(row: dict) -> dict[str, np.ndarray]:
    return {name: np.load(resolve_saved_path(path)) for name, path in row['array_paths'].items()}


def points_inside_shape(points_zyx: np.ndarray, shape_zyx: tuple[int, int, int]) -> bool:
    pts = np.asarray(points_zyx, dtype=float)
    shape = np.asarray(shape_zyx, dtype=float)
    return bool(np.all((pts >= 0.0) & (pts <= shape[None, :] - 1.0)))


def landmark_mask_fraction(mask: np.ndarray, landmarks_zyx: np.ndarray) -> float:
    idx = np.rint(landmarks_zyx).astype(int)
    valid = (
        (idx[:, 0] >= 0) & (idx[:, 0] < mask.shape[0])
        & (idx[:, 1] >= 0) & (idx[:, 1] < mask.shape[1])
        & (idx[:, 2] >= 0) & (idx[:, 2] < mask.shape[2])
    )
    if not np.any(valid):
        return float('nan')
    return float(np.mean(mask[idx[valid, 0], idx[valid, 1], idx[valid, 2]] > 0))


def markdown_table(table_rows: list[dict], columns: list[str]) -> None:
    lines = ['|' + '|'.join(columns) + '|', '|' + '|'.join(['---'] * len(columns)) + '|']
    for row in table_rows:
        lines.append('|' + '|'.join(str(row.get(col, '')) for col in columns) + '|')
    display(Markdown('\n'.join(lines)))


In [ ]:
audit_table = []
all_notebook_checks_pass = True

for row in rows:
    arr = load_case_arrays(row)
    fixed_crop = arr['fixed_T00_crop_ptvintensity_zyx']
    moving_crop = arr['moving_T50_crop_ptvintensity_zyx']
    fixed_reg = arr['fixed_T00_reg1mm_trilinear_zyx']
    moving_reg = arr['moving_T50_reg1mm_trilinear_zyx']
    fixed_mask_crop = arr['fixed_T00_lungmask_crop_zyx']
    moving_mask_crop = arr['moving_T50_lungmask_crop_zyx']
    fixed_mask_reg = arr['fixed_T00_lungmask_reg1mm_nearest_zyx']
    moving_mask_reg = arr['moving_T50_lungmask_reg1mm_nearest_zyx']
    fixed_lm_crop = arr['landmarks_fixed_T00_crop_zyx']
    moving_lm_crop = arr['landmarks_moving_T50_crop_zyx']
    fixed_lm_reg = arr['landmarks_fixed_T00_reg1mm_zyx']
    moving_lm_reg = arr['landmarks_moving_T50_reg1mm_zyx']

    shape_match = (
        fixed_crop.shape == fixed_mask_crop.shape
        and moving_crop.shape == moving_mask_crop.shape
        and fixed_reg.shape == fixed_mask_reg.shape
        and moving_reg.shape == moving_mask_reg.shape
    )
    crop_inside = points_inside_shape(fixed_lm_crop, fixed_crop.shape) and points_inside_shape(moving_lm_crop, moving_crop.shape)
    reg_inside = points_inside_shape(fixed_lm_reg, fixed_reg.shape) and points_inside_shape(moving_lm_reg, moving_reg.shape)
    legacy_shift = np.asarray(row['legacy_minus_ptvreg_bounds_zyx'])
    legacy_shift_expected = bool(np.all(legacy_shift == 1))
    notebook_pass = bool(shape_match and crop_inside and reg_inside and legacy_shift_expected and row['verification']['pass'])
    all_notebook_checks_pass = all_notebook_checks_pass and notebook_pass

    audit_table.append({
        'case': f"{int(row['case_id']):02d}",
        'crop zyx': tuple(fixed_crop.shape),
        'reg zyx': tuple(fixed_reg.shape),
        'legacy-task4 minus pTVReg': legacy_shift.tolist(),
        'crop LM inside': crop_inside,
        'reg LM inside': reg_inside,
        'fixed mask LM frac': f"{landmark_mask_fraction(fixed_mask_reg, fixed_lm_reg):.3f}",
        'moving mask LM frac': f"{landmark_mask_fraction(moving_mask_reg, moving_lm_reg):.3f}",
        'PASS': notebook_pass,
    })

markdown_table(
    audit_table,
    ['case', 'crop zyx', 'reg zyx', 'legacy-task4 minus pTVReg', 'crop LM inside', 'reg LM inside', 'fixed mask LM frac', 'moving mask LM frac', 'PASS'],
)
print('NOTEBOOK_VERIFICATION_PASS', all_notebook_checks_pass)
assert all_notebook_checks_pass


In [ ]:
mask_cmap_fixed = ListedColormap([(0, 0, 0, 0), (0.0, 0.85, 0.35, 0.32)])
mask_cmap_moving = ListedColormap([(0, 0, 0, 0), (0.75, 0.25, 1.0, 0.32)])


def normalize_slice(slice_2d: np.ndarray) -> np.ndarray:
    arr = np.asarray(slice_2d, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr)
    lo, hi = np.percentile(finite, [1, 99])
    if hi <= lo:
        lo, hi = float(finite.min()), float(finite.max())
    if hi <= lo:
        return np.zeros_like(arr)
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def bbox_center_y(mask_a: np.ndarray, mask_b: np.ndarray, fallback_shape: tuple[int, int, int]) -> int:
    union = np.logical_or(mask_a > 0, mask_b > 0)
    coords = np.argwhere(union)
    if coords.size == 0:
        return int(fallback_shape[1] // 2)
    return int(round((coords[:, 1].min() + coords[:, 1].max()) / 2.0))


def draw_orientation_labels(ax) -> None:
    box = dict(facecolor='black', alpha=0.35, pad=1, edgecolor='none')
    ax.text(0.50, 0.985, 'Superior', ha='center', va='top', color='white', fontsize=7, transform=ax.transAxes, bbox=box)
    ax.text(0.50, 0.015, 'Inferior', ha='center', va='bottom', color='white', fontsize=7, transform=ax.transAxes, bbox=box)
    ax.text(0.015, 0.50, 'Patient R', ha='left', va='center', color='white', fontsize=7, rotation=90, transform=ax.transAxes, bbox=box)
    ax.text(0.985, 0.50, 'Patient L', ha='right', va='center', color='white', fontsize=7, rotation=90, transform=ax.transAxes, bbox=box)


def draw_coronal_panel(
    ax,
    volume: np.ndarray,
    mask: np.ndarray,
    landmarks_zyx: np.ndarray,
    y_idx: int,
    spacing_zyx: tuple[float, float, float],
    title: str,
    cmap,
    slab: float,
) -> None:
    y_idx = int(np.clip(y_idx, 0, volume.shape[1] - 1))
    img = np.flip(volume[:, y_idx, :], axis=0)
    msk = np.flip((mask[:, y_idx, :] > 0).astype(np.uint8), axis=0)
    aspect = float(spacing_zyx[0]) / float(spacing_zyx[2])
    ax.imshow(normalize_slice(img), cmap='gray', origin='lower', aspect=aspect, interpolation='nearest')
    ax.imshow(np.ma.masked_where(msk == 0, msk), cmap=cmap, origin='lower', aspect=aspect, interpolation='nearest', vmin=0, vmax=1)

    pts = np.asarray(landmarks_zyx, dtype=float)
    near = pts[np.abs(pts[:, 1] - y_idx) <= slab]
    if near.size:
        x = near[:, 2]
        z_display = (volume.shape[0] - 1) - near[:, 0]
        ax.scatter(x, z_display, s=14, facecolors='none', edgecolors='yellow', linewidths=0.7)
    ax.set_title(f'{title}\ny={y_idx}; LM slab n={near.shape[0]}', fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    draw_orientation_labels(ax)


In [ ]:
saved_figures = []

for row in rows:
    arr = load_case_arrays(row)
    native_spacing = tuple(float(v) for v in row['native_spacing_zyx_mm'])
    reg_spacing = (1.0, 1.0, 1.0)

    fixed_native = arr['fixed_T00_native_ptvintensity_zyx']
    moving_native = arr['moving_T50_native_ptvintensity_zyx']
    fixed_crop = arr['fixed_T00_crop_ptvintensity_zyx']
    moving_crop = arr['moving_T50_crop_ptvintensity_zyx']
    fixed_reg = arr['fixed_T00_reg1mm_trilinear_zyx']
    moving_reg = arr['moving_T50_reg1mm_trilinear_zyx']
    fixed_mask_native = arr['fixed_T00_lungmask_native_zyx']
    moving_mask_native = arr['moving_T50_lungmask_native_zyx']
    fixed_mask_crop = arr['fixed_T00_lungmask_crop_zyx']
    moving_mask_crop = arr['moving_T50_lungmask_crop_zyx']
    fixed_mask_reg = arr['fixed_T00_lungmask_reg1mm_nearest_zyx']
    moving_mask_reg = arr['moving_T50_lungmask_reg1mm_nearest_zyx']
    fixed_lm_native = arr['landmarks_fixed_T00_native0_zyx']
    moving_lm_native = arr['landmarks_moving_T50_native0_zyx']
    fixed_lm_crop = arr['landmarks_fixed_T00_crop_zyx']
    moving_lm_crop = arr['landmarks_moving_T50_crop_zyx']
    fixed_lm_reg = arr['landmarks_fixed_T00_reg1mm_zyx']
    moving_lm_reg = arr['landmarks_moving_T50_reg1mm_zyx']

    y_native = bbox_center_y(fixed_mask_native, moving_mask_native, fixed_native.shape)
    y_crop = bbox_center_y(fixed_mask_crop, moving_mask_crop, fixed_crop.shape)
    y_reg = bbox_center_y(fixed_mask_reg, moving_mask_reg, fixed_reg.shape)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)
    cid = int(row['case_id'])
    fig.suptitle(
        f"4DCT case {cid:02d} | pTVReg crop/index + trilinear 1 mm | "
        f"crop {tuple(row['crop_shape_zyx'])} -> reg {tuple(row['registration_shape_zyx'])}",
        fontsize=11,
    )
    draw_coronal_panel(axes[0, 0], fixed_native, fixed_mask_native, fixed_lm_native, y_native, native_spacing, 'T00 original + native mask', mask_cmap_fixed, 4.0)
    draw_coronal_panel(axes[1, 0], moving_native, moving_mask_native, moving_lm_native, y_native, native_spacing, 'T50 original + native mask', mask_cmap_moving, 4.0)
    draw_coronal_panel(axes[0, 1], fixed_crop, fixed_mask_crop, fixed_lm_crop, y_crop, native_spacing, 'T00 corrected pTVReg crop', mask_cmap_fixed, 4.0)
    draw_coronal_panel(axes[1, 1], moving_crop, moving_mask_crop, moving_lm_crop, y_crop, native_spacing, 'T50 corrected pTVReg crop', mask_cmap_moving, 4.0)
    draw_coronal_panel(axes[0, 2], fixed_reg, fixed_mask_reg, fixed_lm_reg, y_reg, reg_spacing, 'T00 1 mm trilinear + mask', mask_cmap_fixed, 5.0)
    draw_coronal_panel(axes[1, 2], moving_reg, moving_mask_reg, moving_lm_reg, y_reg, reg_spacing, 'T50 1 mm trilinear + mask', mask_cmap_moving, 5.0)

    fig_path = FIGURE_DIR / f'4dct_case{cid:02d}_ptvprep_visual_audit.png'
    fig.savefig(fig_path, dpi=150)
    saved_figures.append(str(fig_path))
    display(fig)
    plt.close(fig)

print(f'Saved {len(saved_figures)} figures to {FIGURE_DIR}')
